# Local Geary: inspect differences around every unit

This lab uses the same synthetic ring graph as the Local Moran notebook. Here we focus on each unit's squared differences from its neighbours. We will calculate local Geary values, inspect the cluster/outlier labels and apply an FDR cutoff to the permutation p-values.

**You need:** Python, NumPy, Matplotlib, libpysal and esda.

The graph is synthetic, not a geographic map. The point is to isolate the local statistic and the way it responds to the neighbourhood definition.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from libpysal.weights import W
from esda import Geary_Local, fdr


## 1. Build the neighbour graph

Twelve units form a ring. Each unit is connected to the unit before and after it. The graph remains fixed as we change the observed values.

In [ ]:
n = 12
neighbors = {i: [(i - 1) % n, (i + 1) % n] for i in range(n)}
w = W(neighbors)
w.transform = 'r'

values = np.array([10, 11, 9, 2, 1, 2, 8, 9, 10, 1, 2, 1], dtype=float)

local = Geary_Local(
    connectivity=w,
    labels=True,
    sig=0.05,
    permutations=999,
    seed=12345,
    alternative='two-sided',
).fit(values)

print('Observed values:', values.tolist())
print('Local Geary values:', np.round(local.localG, 3))
print('Permutation p-values:', np.round(local.p_sim, 3))


In [ ]:
angles = np.linspace(0, 2 * np.pi, n, endpoint=False)
xy = np.column_stack([np.cos(angles), np.sin(angles)])
fig, ax = plt.subplots(figsize=(6, 6))
for i, linked in neighbors.items():
    for j in linked:
        if i < j:
            ax.plot([xy[i, 0], xy[j, 0]], [xy[i, 1], xy[j, 1]], linewidth=2)
ax.scatter(xy[:, 0], xy[:, 1], s=850, zorder=3)
for i, (x, y) in enumerate(xy):
    ax.text(x, y, f'U{i}\n{values[i]:g}', ha='center', va='center', color='white', weight='bold')
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Synthetic ring graph: unit ID and observed value')
plt.show()


## 2. Compare local differences

A smaller local Geary value means the focal unit is more similar to its neighbours. A larger value means it is more different. The statistic alone is not enough: we also inspect the conditional permutation p-value.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(np.arange(n), local.localG)
ax.set_xticks(np.arange(n))
ax.set_xticklabels([f'U{i}' for i in range(n)])
ax.set_xlabel('Spatial unit')
ax.set_ylabel('Local Geary value')
ax.set_title('Local squared-difference statistic by unit')
plt.show()


## 3. Apply a multiple-testing correction

The estimator's `labs` labels use the raw significance threshold passed at construction. They are not automatically adjusted for the number of units being tested. We now calculate an FDR cutoff and use it to create a second set of labels.

In [ ]:
fdr_cutoff = fdr(local.p_sim, alpha=0.05)
passes_fdr = local.p_sim <= fdr_cutoff

label_names = {1: 'Outlier', 2: 'Cluster', 3: 'Other', 4: 'Not significant'}
raw_labels = local.labs.copy()
fdr_labels = raw_labels.copy()
fdr_labels[~passes_fdr] = 4

print('FDR cutoff:', fdr_cutoff)
print('Unit | Local Geary | p-value | Raw label | FDR label')
for i in range(n):
    print(
        f'U{i:02d} | {local.localG[i]:.3f} | {local.p_sim[i]:.3f} | '
        f'{label_names[int(raw_labels[i])]:16s} | {label_names[int(fdr_labels[i])]}'
    )


## 4. Change the values, keep the graph

Try replacing the sequence with an alternating high-low pattern. The graph stays exactly the same. Re-run the calculation and compare the local Geary values. Which units now have the largest differences from their neighbours?

In [ ]:
alternating = np.array([10, 1] * 6, dtype=float)
local_alternating = Geary_Local(
    connectivity=w,
    labels=True,
    permutations=999,
    seed=12345,
    alternative='two-sided',
).fit(alternating)

print('Alternating values:', alternating.tolist())
print('Local Geary values:', np.round(local_alternating.localG, 3))
print('Permutation p-values:', np.round(local_alternating.p_sim, 3))


## What to take away

- Local Geary compares the focal value directly with neighbouring values.
- Small local values indicate similarity; large values indicate local differences.
- Conditional permutations provide a reference distribution for each unit.
- Local tests create a multiple-comparison problem; raw labels and FDR-adjusted labels are not the same.
- Local Geary complements Local Moran rather than replacing it.

Continue with the [Local Geary lesson](../algorithms/statistics/local-geary.md), [Global Geary's C](../algorithms/statistics/gearys-c.md) and the [global spatial autocorrelation learning path](../learning-paths/global-spatial-autocorrelation.md).

**Reference:** [PySAL's Local Geary guide](https://pysal.org/esda/stable/user-guide/local_geary.html) and [API reference](https://pysal.org/esda/stable/generated/esda.Geary_Local.html).